# Architecture Pipeline Guide
## How to write good flexible code

In [ ]:
from dataclasses import dataclass

## CLI flags become a config
cli flags are arguments you put when running main.
you can say main.py --controller pid:fast -- vision real:default

you basically are setting what type (of controller, of vision, of anything in the project) you want to use and with what preset

In [ ]:
# before running anything, we build the config from cli flags
config = {
    "controller": "pid:fast",
    "vision": "sim_cam:default",
    "dt": 0.004,
    ...
}
# it establishes the type of experiment we will do

## Classes
Every class has two things, presets dict and params dataclass.

Dict has all presets to this class, maybe you want a faster controller, different poles, different estimator

Every dataclass groups what is in the dict into an organized object to sent to the class itself.

In [ ]:
# class presets, many of them to choose, needs default
PID_PRESETS = {
    "default": {"kp": 1.0, "ki": 0.2, "kd": 0.05},
    "fast":    {"base": "default", "kp": 2.0},
}

# dataclass of init params for each class, you add a param here, you only change it in default preset
@dataclass
class PIDParams:
    kp: float
    ki: float
    kd: float

You basically want each class to receive a single thing

In [ ]:
class PIDController():
    def __init__(self, params: PIDParams):
        self.kp = params.kp
        self.ki = params.ki
        self.kd = params.kd

When you want to choose a dict, it needs to become into a dataclass to be able to be sent to the class:

In [ ]:
# create right preset if we have base
def resolve_preset(presets, name):
    p = presets[name]
    if "base" in p:
        base = resolve_preset(presets, p["base"])
        return {**base, **{k: v for k, v in p.items() if k != "base"}}
    return p

# init class, each class needs their builder, they all use resolve_preset.
def build_pid(preset):
    raw = resolve_preset(PID_PRESETS, preset)
    return PIDParams(**raw)

### Spec
Every builder connects a preset dict to params dataclass. This makes initializing classes easy, because you just go

```
params = builder(preset)
class(params)
```
We need *preset*, *builder*, and *class*, with those three things we can initialize a class.

*Note*: config gives us type:preset

where do we get the *preset*? from the config

where do we get the *builder* and *class*? from the Spec, which the registry can give us based on the type from config

In [ ]:
@dataclass
class Spec:
    cls: type
    builder: callable
    sim_only: bool | None = None

CONTROLLER_REGISTRY = {
    "pid": Spec(PIDController, build_pid),
    "lqr": Spec(LQRController, build_lqr),
}

# Building
If each classes registry has a Spec object with the type and builder, then building is as easy as having the registry and the config.

If an object needs params only, this works. If an object's params include other objects, then that's when we need a factory to assemble objects first.

In [ ]:
def build_system(config):
    controller = build_from_registry(CONTROLLER_REGISTRY, config["controller"])
    estimator  = build_from_registry(ESTIMATOR_REGISTRY, config["estimator"])
    vision     = build_from_registry(VISION_REGISTRY, config["vision"])
    actuator   = build_from_registry(ACTUATOR_REGISTRY, config["actuator"])

    return system(controller, estimator, vision, actuator)

def build_from_registry(registry, spec_string):
    type_, preset = spec_string.split(":")
    spec = registry[type_]
    try:
        spec = registry[type_]
    except KeyError:
        raise ValueError(f"Unknown type: {type_}")

    params = spec.builder(preset)
    return spec.cls(params)

# to build multiple classes, create a system builder that builds in chunks
system = build_system(config)

So system is the physical things, but the experiment needs more abstract classes to help do it.

In [ ]:
# maybe have a registry of registries for Experiment? This doesnt work but smth like this? or just manually like before in systems

@dataclass
class ExperimentParams:
    system: type System
    logger: type Logger
    stop_conditions: type StopCondition
    visualizer: type Visualizer
    
EXPERIMENT_REGISTRY = {
    "system": SYSTEM_REGISTRY,
    "logger": LOGGER_REGISTRY,
    ...
}


def build_experiment(config):
    raw = []
    
    for type_ in config:
        if registry[type_]:
            type_obj = build_from_registry(registry[type_], type_)
            raw.append(type_obj)

    return Experiment(**raw)

# Main 
Main just sets all the cli presets and overrides an builds from factories and runs it.

In [ ]:

   
def main():
    # 1. Parse CLI
    args = parse_args()

    # 2. Resolve config (system preset + overrides)
    config = CONFIG_PRESETS[args.config].copy()

    if args.controller:
        config["controller"] = args.controller
    if args.estimator:
        config["estimator"] = args.estimator
    if args.camera:
        config["camera"] = args.camera
    if args.n_runs:
        config["n_runs"] = args.n_runs
    #one for each possible override

    # 3. Build experiment (factory)
    experiment = build_experiment(config)

    # 4. Build runner
    scheduler = build_from_registry(SCHEDULER_REGISTRY, config["scheduler"])
    runner = Runner(experiment, scheduler)

    # 5. Run experiments
    all_results = []

    n_runs = config["n_runs"]
    for i in range(n_runs):
        print(f"Run {i+1}/{n_runs}")

        runner.exp.reset()
        runner.run_once(T=args.T, dt=args.dt)

        data = runner.exp.logger.get_data()
        all_results.append(data)

    # 6. Aggregate results
    summary = summarize_results(all_results)

    # 7. Print / save
    print_summary(summary)
    # save_results(all_results, summary)

## Config Presets

Assembling a config from CLI flags can be tedius, so we can build some presets that assemble one for us

If we want to change one thing, it's as easy as calling the preset and overriding what we want

```
main.py --config sim --controller lqr:default
```


In [ ]:
CONFIG_PRESETS = {
    "sim": {
        "controller": "pid:default",
        "vision": "sim_cam:default",
        "dt": 0.002,
    }
}

## Incompatibility

The only thing that breaks stuff, is when we try to do an offline experiment with online classes. Montecarlo with real_dvs_cams? No can do. That's what the Spec sim_only attribute is used for. Most leave it as None, but actuator and vision use it to determine if an experiment can be run offline and therefore can be used in montecarlo simulations. You check it like so

In [ ]:
def is_fully_sim(config):
    specs = [
        # get specs of the type of actuator we have
        ACTUATOR_REGISTRY[get_type(config["actuator"])],
        VISION_REGISTRY[get_type(config["vision"])],
        # add others if needed
    ]

    # determine if all types chosen are sim=True or not
    return all(spec.sim_only for spec in specs)

def get_type(spec_str:str):
    # config gives us type:preset, return type only
    return spec_str.split(":")[0]

## Changing config

You'll be tempted to run a config preset and change some params. Go ahead, but this is a dev tool. Users in the UI are not allowed to override anything other than controller and estimator because those two are compatible with all configs.

An example of being careful is using real_vision preset. this makes mock servo, connect to dvs_cams, and runs in realtime with a visualizer too. If you want to use sim_dvs, you can, and it will work essentially like a realtime_sim_dvs preset, but you won't have to remember or create that config preset. If you had used sim preset, you would have to change the actuator, the realtime scheduler, the visualization in realtime, turn off animation after experiment. Doable but more work. If this happens often, then creating a new config preset might be a good idea. Like real_vision and realtime_sim. 

# State Machine

We have gotten to the point of realism where I want to include a state machine, the path goes as follows